Joining static datas into one

In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
wind_farm_elevation =  gpd.read_parquet(
    "../data/interim/terrain_processed/wind_farm_elevation.parquet"
)

In [ ]:
wind_cornie = gpd.read_parquet(
    "../data/interim/cornie_processed/wind_cornie.parquet"
)

In [ ]:
#joining wind farm with elevation and cornie i.e. vegetation
wind_farms_spatial = (
    wind_farm_elevation
    .merge(
        wind_cornie[
            [
                "Site Name",
                "Installed Capacity (MWelec)",
                "Code_18",
                "landcover_status"
            ]
        ],
        on=[
            "Site Name",
            "Installed Capacity (MWelec)"
        ],
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
wind_farms_spatial.head()

In [ ]:
import xarray as xr
#weather variables
weather_feature = xr.open_dataset(
    "../data/interim/era5_processed/weather_feature.nc"
)

In [ ]:
farm_grid_mapping = []

#joining farm gird mapping with ERA5 cells by lat and lon
for _, row in wind_farms_spatial.iterrows():

    point = row.geometry

    lon = point.x
    lat = point.y

    nearest = weather_feature.sel(
        latitude=lat,
        longitude=lon,
        method="nearest"
    )

    farm_grid_mapping.append({
        "Site Name": row["Site Name"],
        "Installed Capacity (MWelec)":
            row["Installed Capacity (MWelec)"],

        "farm_lat": lat,
        "farm_lon": lon,

        "era5_lat": float(nearest.latitude),

        "era5_lon": float(nearest.longitude)
    })

In [ ]:
farm_grid_mapping = pd.DataFrame(
    farm_grid_mapping
)

In [ ]:
farm_grid_mapping.head()

In [ ]:
#farm_grid_mapping saved as parquet

Searching for duplicates and observing them

In [ ]:
point = wind_farms_spatial.geometry.iloc[0]

lon = point.x
lat = point.y

In [ ]:
farm_grid_mapping[
    ["era5_lat", "era5_lon"]
].drop_duplicates().shape

In [ ]:
farm_grid_mapping[
    "Installed Capacity (MWelec)"
].describe()

In [ ]:
#example showing that one same era5 cell consits many wind farms
farm_grid_mapping[
    farm_grid_mapping["era5_lat"] == 50.25
][
    ["Site Name", "Installed Capacity (MWelec)"]
]

In [ ]:
#changing data type
farm_grid_mapping["Installed Capacity (MWelec)"] = pd.to_numeric(
    farm_grid_mapping[
        "Installed Capacity (MWelec)"
    ],
    errors="coerce"
)

In [ ]:
#grouping wind farm falling in same era5 cell
grid_capacity = (
    farm_grid_mapping
    .groupby(
        ["era5_lat", "era5_lon"]
    )
    ["Installed Capacity (MWelec)"]
    .sum()
    .reset_index()
)

In [ ]:
grid_capacity[
    "Installed Capacity (MWelec)"
].sort_values(ascending=False).head(20)

In [ ]:
grid_capacity.to_parquet(
    "../data/interim/weather_processed/grid_capacity.parquet",
    index=False
)

<h1>Simple UK Mean Weather

In [ ]:
#datasets: weather_feature.nc

In [ ]:
mean_weather_feature = weather_feature.mean(dim=["latitude", "longitude"])

In [ ]:
mean_weather_feature = mean_weather_feature.to_dataframe().reset_index()

In [ ]:
mean_weather_feature = mean_weather_feature.rename(columns={'valid_time':'timestamp'})

In [ ]:
mean_weather_feature = mean_weather_feature.merge(
    wind_generation_hourly,
    on='timestamp',
    how='inner'
)

In [ ]:
mean_weather_feature = mean_weather_feature.drop(columns=["number", "expver"])

In [ ]:
mean_weather_feature.to_parquet(
    "../data/processed/mean_weather_feature.parquet",
    index=False
)

<h1> Wind Farm Mean Weather

In [ ]:
#datasets: grid_capacity.parquet and weather_feature.nc

In [ ]:
grid_capacity.columns

In [ ]:
filtered_cells = []

#selecting only wind-farm era5 cells
for _, row in grid_capacity.iterrows():
    cell = weather_feature.sel(
        latitude = row["era5_lat"],
        longitude=row["era5_lon"]
    )

    filtered_cells.append(cell)

#combine all selected cells
wf_ds = xr.concat(filtered_cells, dim="gridcell")

#equal weight avevrage
wf_mean_ds = wf_ds.mean(dim="gridcell")

#to dataframe
wf_mean_ds = (
    wf_mean_ds
    .to_dataframe()
    .reset_index()
)


In [ ]:
wf_mean_ds = wf_mean_ds.rename(columns={'valid_time':'timestamp'})

In [ ]:
wf_mean_ds = wf_mean_ds.merge(
    wind_generation_hourly,
    on='timestamp',
    how='inner'
)

In [ ]:
wf_mean_ds = wf_mean_ds.drop(columns=["number", "expver"])

In [ ]:
wf_mean_ds.head()

In [ ]:
wf_mean_ds.to_parquet(
    "../data/processed/wf_mean_ds.parquet",
    index=False
)

<h1>Capacity weighted the atmospheric variables

Where weight = grid capacity / total capacity

In [ ]:
#total capacity
total_capacity = (
    grid_capacity[
        "Installed Capacity (MWelec)"
    ].sum()
)

In [ ]:
#grid capcity weight
grid_capacity["weight"] = (
    grid_capacity[
        "Installed Capacity (MWelec)"
    ]
    /total_capacity
)

In [ ]:
grid_capacity.head()

In [ ]:
grid_capacity.isna().sum()

In [ ]:
weighted_weather = None

In [ ]:
#giving weight values to weather features as well
for _, row in grid_capacity.iterrows():

    lat = row["era5_lat"]
    lon = row["era5_lon"]

    weight = row["weight"]

    cell_weather = weather_feature.sel(
        latitude=lat,
        longitude=lon
    )

    cell_weather = cell_weather * weight


    if weighted_weather is None:

        weighted_weather = cell_weather

    else:

        weighted_weather = (
            weighted_weather + cell_weather
        )

        

In [ ]:
weighted_weather_df = (
    weighted_weather
    .to_dataframe()
    .reset_index()
)

In [ ]:
weighted_weather.to_parquet(
    "../data/interim/weather_processed/weighted_weather.parquet",
    index=False
)

joining target variable table with the current weighted dataset

In [ ]:
wind_generation_hourly = pd.read_parquet(
    "../data/interim/neso_processed/wind_generation_hourly.parquet"
)

In [ ]:
wind_generation_hourly = wind_generation_hourly.rename(columns={"DATETIME":"timestamp"})

In [ ]:
weighted_weather["timestamp"].min()
weighted_weather["timestamp"].max()


In [ ]:

wind_generation_hourly["timestamp"].max()


In [ ]:
vws_dataset = weighted_weather.merge(
    wind_generation_hourly,
    on="timestamp",
    how="inner"
)

In [ ]:
vws_dataset.shape

In [ ]:
vws_dataset.isna().sum()

In [ ]:
vws_dataset.head()

In [ ]:
vws_dataset = vws_dataset.drop(columns=["number", "expver", "latitude"])

In [ ]:
vws_dataset.to_parquet(
    "../data/processed/vws_dataset.parquet",
    index=False
)

In [ ]:
%reset